<a href="https://colab.research.google.com/github/Innovatewithapple/transformer-paper-rag/blob/main/transformer_paper_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers faiss-cpu pymupdf

In [ ]:
import numpy as np
import random
import os
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
      torch.cuda.manual_seed(seed)
      torch.cuda.manual_seed_all(seed)

      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

setProductionSeed(True)

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import re
import faiss
from sentence_transformers import SentenceTransformer,CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM,pipeline
import pymupdf
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('huggingfaceToken'))  # your token here

In [ ]:
#----------Load Encoder Model------------#
encoder_model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5',trust_remote_code=True)

#---------Load Decoder Models-----------#
decoder_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", trust_remote_code=True)
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token  # ← add this
llm = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct",dtype=torch.float16,device_map="auto",trust_remote_code=True)

In [ ]:
# ---- LOAD RERANKER (once at startup) ----
# swap model name here anytime — nothing else changes
reranking_model = CrossEncoder('BAAI/bge-reranker-large',device='cuda:0')

In [ ]:

print(decoder_tokenizer.chat_template)

In [ ]:
#------Extract and Clean----------!
def extract_and_clean_pdf(pdf_path):
  doc = pymupdf.open(pdf_path)
  full_text = ""
  for page in doc:
    full_text += page.get_text() + "\n"

  full_text = re.sub(r'-\n', '', full_text) #Removes broken hyphenated words across lines. (transf-ormer = transformer)
  full_text = re.sub(r'\n\d+\n', '\n', full_text) #Removes page numbers.
  full_text = re.sub(r'\[\d+\]', '', full_text) #Removes citation-like patterns.
  full_text = re.sub(r'\n{3,}', '\n\n', full_text) #Removes excessive empty lines.
  full_text = re.sub(r'(?<!\n)\n(?!\n)', ' ', full_text) #This fixes broken sentences. next char or previous char shouldn't be in next line
  full_text = re.sub(r' {2,}', ' ', full_text) #Removes multiple spaces.
  # remove email addresses
  full_text = re.sub(r'\S+@\S+\.\S+', '', full_text)

  # remove lines that are only names/affiliations (short lines before abstract)
  # everything before "Abstract" is likely title page noise
  abstract_pos = full_text.find('Abstract')
  if abstract_pos != -1:
      full_text = full_text[abstract_pos:]  # start from Abstract

  return full_text.strip()

In [ ]:
# ---- SANITY CHECK ----
text = extract_and_clean_pdf('/kaggle/input/datasets/mihirvyas/attentionisallyouneedpaper/NIPS-2017-attention-is-all-you-need-Paper.pdf')
sentences = sent_tokenize(text)
avg_sent = sum(len(s) for s in sentences) // len(sentences)
max_sent = max(len(s) for s in sentences)
min_sent = min(len(s) for s in sentences)

# model limits
embedding_token_limit = 8192
embedding_safe_chars  = embedding_token_limit * 4

llm_token_limit = 32768  # Qwen2.5
top_k = 3
child_size  = 400
parent_size = 1200

print("====== SANITY CHECK ======")
print(f"\n--- DOCUMENT ---")
print(f"Total chars:         {len(text)}")
print(f"Total sentences:     {len(sentences)}")
print(f"Avg sentence length: {avg_sent} chars")
print(f"Max sentence length: {max_sent} chars")
print(f"Min sentence length: {min_sent} chars")

print(f"\n--- EMBEDDING MODEL ---")
print(f"Token limit:         {embedding_token_limit}")
print(f"Safe chunk limit:    {embedding_safe_chars} chars")
print(f"Your child_size:     {child_size} chars")
if child_size < embedding_safe_chars:
    print(f"Status:              ✓ SAFE")
else:
    print(f"Status:              ✗ DANGER")

print(f"\n--- LLM CONTEXT ---")
tokens_to_llm = (top_k * parent_size) // 4
print(f"Tokens sent to LLM:  {tokens_to_llm}")
print(f"LLM token limit:     {llm_token_limit}")
if tokens_to_llm < llm_token_limit * 0.7:
    print(f"Status:              ✓ SAFE")
else:
    print(f"Status:              ✗ DANGER")

print(f"\n--- CHUNK ESTIMATE ---")
expected_chunks = len(text) // child_size
print(f"Expected chunks:     {expected_chunks}")
print(f"If actual >> this:   chunking logic broken")
print("==========================")

In [ ]:
def parent_child_chunk(text,parent_size=1200,child_size=500,overlap=1):
  sentence = sent_tokenize(text)

  #build parents
  parents = []
  current = []
  current_len = 0

  for sent in sentence:
    if current_len + len(sent) < parent_size:
      current.append(sent)
      current_len += len(sent)
    else:
      if current:
        parents.append(" ".join(current))
      current = current[-overlap:] + [sent]
      current_len = sum(len(s) for s in current)

  if current:
    parents.append(" ".join(current))


  #build children
  result = []
  for p_idx,parent in enumerate(parents):
    children = []
    current = []
    current_len = 0

    for sent in sent_tokenize(parent):
      if current_len + len(sent) < child_size:
        current.append(sent)
        current_len += len(sent)
      else:
        if current:
          children.append(" ".join(current))
        current = current[-overlap:] + [sent]
        current_len = sum(len(s) for s in current)

    if current:
      children.append(" ".join(current))

    for c_idx,child in enumerate(children):
      result.append({
          'child_text':child,
          'parent_text':parent,
          'p_idx':p_idx,
          'child_idx':f'{p_idx}_{c_idx}'
      })

  return result

In [ ]:
text = extract_and_clean_pdf('/content/NIPS-2017-attention-is-all-you-need-Paper.pdf')
All_chunks = parent_child_chunk(text=text,parent_size=1200,child_size=600,overlap=1)
print(len(All_chunks))

In [ ]:
#--------Extract child texts for embedding--------!
child_texts = [c['child_text'] for c in All_chunks]
parent_texts = [c['parent_text'] for c in All_chunks]
p_idx = [c['p_idx'] for c in All_chunks]
child_idx = [c['child_idx'] for c in All_chunks]

prefix = ['search_document: '+t for t in child_texts]

embeddings = encoder_model.encode(prefix,batch_size=32,show_progress_bar=True,normalize_embeddings=True)

#-------Build faiss index--------!
dim = embeddings.shape[1] # 786 features per vector

index = faiss.IndexFlatIP(dim) #Index is a vectordb structure inside faiss, and flat just flatten the features and apply dot product/cosine similarity
index.add(embeddings.astype(np.float32))
print(f'fiass index build. total vectors: {index.ntotal}')

In [ ]:
#------Extract child texts for embedding using IVF------!
child_texts = [c['child_text'] for c in All_chunks]
parent_texts = [c['parent_text'] for c in All_chunks]
p_idx = [c['p_idx'] for c in All_chunks]
child_idx = [c['child_idx'] for c in All_chunks]

prefix = ['search_document: '+t for t in child_texts]

embeddings = encoder_model.encode(prefix,batch_size=32,show_progress_bar=True,normalize_embeddings=True)

dim = embeddings.shape[1]
quantizer = faiss.IndexFlatIP(dim)

index = faiss.IndexIVFFlat(quantizer,dim,10,faiss.METRIC_INNER_PRODUCT)
index.train(embeddings.astype(np.float32))
index.add(embeddings.astype(np.float32))

index.nprobe = 3

#nlist means how many clusters need to create (most of the time set value of sqrt(number of total chunks))
#nprobe means from those clusters how many will be search for matching (the more the better but will be slower)

In [ ]:
#------Extract child texts for embedding using HNSW------!
child_texts = [c['child_text'] for c in All_chunks]
parent_texts = [c['parent_text'] for c in All_chunks]
p_idx = [c['p_idx'] for c in All_chunks]
child_idx = [c['child_idx'] for c in All_chunks]

prefix = ['search_document: '+t for t in child_texts]

embeddings = encoder_model.encode(prefix,batch_size=32,show_progress_bar=True,normalize_embeddings=True)

dim = embeddings.shape[1]

index = faiss.IndexHNSWFlat(dim,32)
index.hnsw.efConstruction = 200
index.add(embeddings.astype(np.float32))
index.hnsw.efSearch = 50

#M means how many connections each vector will have(or the close neighbors a vector will have)
#efConstruction means now vector are surrounded by many other vectors, so efConstruction decide which 16 i need to choose. and the more high efconstruction is the more better neighbors it can choose.
#efSearch means how far will it go to search before stop

In [ ]:
#---------Retrieval Method--------!
def Retrieval(question,top_k=5):
  query = ['search_query: '+question]
  query_vector = encoder_model.encode(query,normalize_embeddings=True).astype(np.float32)

  #Search
  scores,indices = index.search(query_vector,top_k)
  results = []

  for score,idx in zip(scores[0],indices[0]):
    results.append({
        'score':float(score),
        'child_text':child_texts[idx],
        'parent_text':parent_texts[idx],
        'child_id':child_idx[idx]
    })
      # clean printing

  # for i, r in enumerate(results):
  #    print("=" * 80)
  #    print(f"Rank      : {i+1}")
  #    print(f"Score     : {r['score']:.4f}")
  #    print(f"Child ID  : {r['child_id']}")

  #    print("\nChild Text:\n")
  #    print(r['child_text'])

  #    print("\nParent Text:\n")
  #    print(r['parent_text'])

  #    print("\n")
  return results

# **CrossEncoder (Reranking)**

In [ ]:
def Reranking(question,results,top_k=2):
    pairs = [(question,r['child_text']) for r in results]
    scores = reranking_model.predict(pairs)
    # print('RetrievedScores: ',scores)

    for i,result in enumerate(results):
        result['rerank_score'] = float(scores[i])

    #Put best scores first
    reranked = sorted(results,key=lambda x:x['rerank_score'],reverse=True) # put true makes high scores first, default put low scores first
    # for r in reranked:
    #  print("="*80)
    #  print("Question: ",question)
    #  print("Rerank Score:", r['rerank_score'])
    #  print("\nText:\n")
    #  print(r['child_text'])
    return reranked[:top_k]

# **Decoder**

In [ ]:
#-------Prompt Builder-------!
def buildPrompt(question,result):
  context = "\n---\n".join([r['child_text'] for r in result])

  message = [
      {"role": "system", "content": """
You are a precise question-answering assistant.

Answer using ONLY the provided context.

Rules:
1. Do NOT use outside knowledge.
2. Copy numerical and decimal values exactly as written in the context.
3. Do NOT round, estimate, modify, or infer numbers.
4. Do NOT explain or expand abbreviations unless explicitly written in the context.
5. Preserve important entities, task names, benchmark names, and relationships exactly as written in the context.
6. Include the specific task or subject associated with the answer if present in the context.
7. Give a clear and natural answer in 1-2 sentences.
8. Keep the answer grounded strictly in the provided context.
9. If the exact answer is not explicitly stated but can be reasonably inferred from the context, provide the most likely answer based ONLY on the context.
10.If the context does not contain enough information, say:
    "I don't know."
"""},
      {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
  ]

  prompt = decoder_tokenizer.apply_chat_template(message,tokenize=False,add_generation_prompt=True)
  return prompt

In [ ]:
def Answer(question):
  result = Retrieval(question,top_k=3)
  top_3 = Reranking(question,result,top_k=1)
  prompt = buildPrompt(question,top_3)

  inputs = decoder_tokenizer(prompt,return_tensors='pt').to(llm.device)
  input_tokens = inputs['input_ids'].shape[1] # input has batch size and token count, we need this count to get answer without question include in text

  #generate
  with torch.no_grad():
    output = llm.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.1,
        repetition_penalty=1.0,
        no_repeat_ngram_size=3,
        top_p=0.9,
        pad_token_id=decoder_tokenizer.eos_token_id
    )

  # decode only the answer part
  answer_tokens = output[0][input_tokens:]
  return decoder_tokenizer.decode(answer_tokens, skip_special_tokens=True)

**Question-Answers******

In [ ]:
questions = [
    "What model architecture is introduced in the paper?",
    "What does the encoder in Transformer consist of?",
    "How many attention heads did they use?",
    "What BLEU score did they achieve on English-German translation?",
    "How long did training take on 8 GPUs?",
    "What is the key advantage over RNN models?",
    "What optimizer did they use?",
    "What is multi-head attention?"
]

In [ ]:
#------Retrieve--------!
for q in questions:
    print(f"Q: {q}")
    result = Answer(q)
    print(f"\nA: {result}")
    print("---\n")


In [ ]:
question = "What optimizer did they use?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

In [ ]:
question = "What BLEU score did they achieve on English-German translation?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

In [ ]:
question = "What does the encoder in Transformer consist of?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

In [ ]:
question = "How many attention heads did they use?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

In [ ]:
question = "How long did training take on 8 GPUs?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

In [ ]:
question = "What is the key advantage over RNN models?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

In [ ]:
question = "What model architecture is introduced in the paper?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")